# HFE-7200 tubing pressure-drop calculator

Corrected simple-physics model for estimating the liquid HFE-7200 pressure drop through the current loop geometry.

## Model and sources

For an incompressible single-phase liquid line, the pressure drop is modeled with Darcy-Weisbach pipe friction plus standard minor losses using the mean velocity head:

$$
\Delta P = \left(f_D\frac{L}{D} + K_\mathrm{minor}\right)\frac{\rho v^2}{2},\qquad v = \frac{Q}{A}.
$$

Here $f_D$ is the Darcy friction factor: a dimensionless wall-friction coefficient used in the Darcy-Weisbach equation. It is not the Fanning friction factor; for the same flow, $f_D = 4 f_\mathrm{Fanning}$. For laminar pipe flow, $f_D=64/Re$. A smooth-pipe Blasius fallback is included for higher Reynolds numbers, but the nominal design point here is laminar.

The Reynolds number can be computed either from dynamic viscosity, $Re=\rho vD/\mu$, or from kinematic viscosity, $Re=vD/\nu$, because $\nu=\mu/\rho$. This notebook starts from the documented kinematic-viscosity table, converts to dynamic viscosity with $\mu=\rho\nu$, and uses the dynamic-viscosity form; the two forms are checked to give the same value.

Here $K$ is a dimensionless loss coefficient. It says how many mean-velocity heads, $\rho v^2/2$, are lost by a fitting or pipe section. The pipe contribution is $K_\mathrm{pipe}=f_D L/D$; the aggregate fitting contribution is $K_\mathrm{minor}$ from bends, valves, contractions, and expansions. Larger $K$ means larger pressure drop at the same flow and diameter.

Fluid-property sources:

- Density: `analysis/docs/HFE 7200 Technical Data.pdf`, $\rho = 1000(1.4811 - 0.0023026T_C)$ kg/m$^3$.
- Kinematic viscosity: HFE-7200 table used in `analysis/notebooks/HFE_properties.ipynb`, transcribed from `analysis/docs/HFE Freezing Data.pdf`.

The old notebook used a fixed $\mu=6$ mPa s. At the nominal $-110^\circ$C point used below, log interpolation of the documented viscosity table gives $\nu=28.35$ cSt and $\mu=\rho\nu=49.18$ mPa s.

In [1]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PSI_PER_PA = 1.0 / 6894.757293168
PA_PER_BAR = 1.0e5


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'analysis' / 'docs').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root containing analysis/docs and data.')


REPO_ROOT = find_repo_root()

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.28,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})


print(f'Repo root: {REPO_ROOT}')
print('Figure output: inline notebook display only')

Repo root: /home/aamy/Documents/hfe-system
Figure output: inline notebook display only


In [2]:
@dataclass(frozen=True)
class LineGeometry:
    length_m: float = 3.75
    od_in: float = 3 / 8
    wall_in: float = 0.035
    n_90: int = 6
    n_45: int = 0
    n_ball_valves: int = 1
    n_contractions: int = 1
    n_expansions: int = 1
    beta_contraction: float = 0.001
    beta_expansion: float = 0.001


@dataclass(frozen=True)
class HFE7200State:
    temperature_C: float = -110.0


@dataclass(frozen=True)
class LossCoefficients:
    K_90: float = 0.65
    K_45: float = 0.35
    K_ball: float = 0.04


HFE7200_VISCOSITY_TABLE = pd.DataFrame(
    {
        'temperature_C': [25.0, 0.0, -10.0, -20.0, -30.0, -40.0, -50.0, -60.0, -70.0, -100.0, -120.0],
        'kinematic_viscosity_cSt': [0.41, 0.67, 0.78, 0.93, 1.14, 1.42, 1.84, 2.48, 3.72, 12.47, 64.47],
    }
).sort_values('temperature_C')


def id_from_od_wall(od_in: float, wall_in: float) -> float:
    return (od_in - 2.0 * wall_in) * 0.0254


def hfe7200_density_kg_m3(temperature_C):
    temperature_C = np.asarray(temperature_C, dtype=float)
    return 1000.0 * (1.4811 - 0.0023026 * temperature_C)


def hfe7200_kinematic_viscosity_cst(temperature_C):
    """Return HFE-7200 kinematic viscosity by log interpolation of the reference table."""

    temperature_C = np.asarray(temperature_C, dtype=float)
    table_temperature = HFE7200_VISCOSITY_TABLE['temperature_C'].to_numpy(float)
    table_log_nu = np.log(HFE7200_VISCOSITY_TABLE['kinematic_viscosity_cSt'].to_numpy(float))

    if np.nanmin(temperature_C) < table_temperature.min() or np.nanmax(temperature_C) > table_temperature.max():
        raise ValueError(
            'Viscosity interpolation is only supported over '
            f'{table_temperature.min():.0f}..{table_temperature.max():.0f} C.'
        )

    return np.exp(np.interp(temperature_C, table_temperature, table_log_nu))


def hfe7200_dynamic_viscosity_pa_s(temperature_C):
    rho = hfe7200_density_kg_m3(temperature_C)
    nu_m2_s = hfe7200_kinematic_viscosity_cst(temperature_C) * 1.0e-6
    return rho * nu_m2_s


def darcy_friction_factor(Re):
    Re = np.asarray(Re, dtype=float)
    if np.any(Re <= 0.0):
        raise ValueError('Reynolds number must be positive.')

    laminar = 64.0 / Re
    turbulent = 0.3164 / Re**0.25  # Blasius smooth-pipe approximation.
    transition_weight = np.clip((Re - 2300.0) / (4000.0 - 2300.0), 0.0, 1.0)
    return (1.0 - transition_weight) * laminar + transition_weight * turbulent


def minor_loss_coefficient(geometry: LineGeometry, losses: LossCoefficients) -> float:
    return (
        geometry.n_90 * losses.K_90
        + geometry.n_45 * losses.K_45
        + geometry.n_ball_valves * losses.K_ball
        + geometry.n_contractions * 0.45 * (1.0 - geometry.beta_contraction)
        + geometry.n_expansions * (1.0 - geometry.beta_expansion) ** 2
    )


def pressure_drop_hfe7200(
    flow_L_min,
    *,
    geometry: LineGeometry = LineGeometry(),
    state: HFE7200State = HFE7200State(),
    losses: LossCoefficients = LossCoefficients(),
    diameter_m=None,
    minor_loss_scale: float = 1.0,
) -> pd.DataFrame:
    if diameter_m is None:
        diameter_m = id_from_od_wall(geometry.od_in, geometry.wall_in)

    flow_m3_s = np.asarray(flow_L_min, dtype=float) * 1.0e-3 / 60.0
    diameter_m = np.asarray(diameter_m, dtype=float)
    flow_m3_s, diameter_m = np.broadcast_arrays(flow_m3_s, diameter_m)
    flow_m3_s = np.ravel(flow_m3_s)
    diameter_m = np.ravel(diameter_m)

    rho = float(hfe7200_density_kg_m3(state.temperature_C))
    nu_cst = float(hfe7200_kinematic_viscosity_cst(state.temperature_C))
    mu_pa_s = float(hfe7200_dynamic_viscosity_pa_s(state.temperature_C))
    nu_m2_s = nu_cst * 1.0e-6

    area_m2 = np.pi * diameter_m**2 / 4.0
    velocity_m_s = flow_m3_s / area_m2
    Re = rho * velocity_m_s * diameter_m / mu_pa_s
    Re_from_kinematic = velocity_m_s * diameter_m / nu_m2_s
    if not np.allclose(Re, Re_from_kinematic, rtol=1.0e-12, atol=0.0):
        raise AssertionError('Dynamic- and kinematic-viscosity Reynolds forms disagree.')
    f_D = darcy_friction_factor(Re)

    K_pipe = f_D * geometry.length_m / diameter_m
    K_minor = minor_loss_scale * minor_loss_coefficient(geometry, losses)
    K_total = K_pipe + K_minor
    velocity_head_pa = rho * velocity_m_s**2 / 2.0
    dP_Pa = K_total * velocity_head_pa

    return pd.DataFrame(
        {
            'flow_L_min': flow_m3_s * 60.0 * 1000.0,
            'temperature_C': state.temperature_C,
            'density_kg_m3': rho,
            'kinematic_viscosity_cSt': nu_cst,
            'dynamic_viscosity_mPa_s': mu_pa_s * 1000.0,
            'diameter_m': diameter_m,
            'velocity_m_s': velocity_m_s,
            'Re': Re,
            'darcy_f': f_D,
            'K_pipe': K_pipe,
            'K_minor': K_minor,
            'K_total': K_total,
            'dP_Pa': dP_Pa,
            'dP_bar': dP_Pa / PA_PER_BAR,
            'dP_psi': dP_Pa * PSI_PER_PA,
        }
    )


def pressure_drop_envelope(
    diameters_m,
    *,
    flow_L_min: float = 4.0,
    geometry: LineGeometry = LineGeometry(),
    state: HFE7200State = HFE7200State(),
    minor_loss_scale_bounds: tuple[float, float] = (0.7, 1.3),
) -> pd.DataFrame:
    """Return a deterministic sensitivity band from aggregate minor-loss K uncertainty only."""

    diameters_m = np.asarray(diameters_m, dtype=float)
    pressure_samples = []

    for minor_loss_scale in minor_loss_scale_bounds:
        pressure_samples.append(
            pressure_drop_hfe7200(
                flow_L_min,
                geometry=geometry,
                state=state,
                diameter_m=diameters_m,
                minor_loss_scale=float(minor_loss_scale),
            )['dP_bar'].to_numpy(float)
        )

    pressure_samples = np.vstack(pressure_samples)
    return pd.DataFrame(
        {
            'diameter_m': diameters_m,
            'dP_low_bar': np.nanmin(pressure_samples, axis=0),
            'dP_high_bar': np.nanmax(pressure_samples, axis=0),
        }
    )

In [3]:
geometry = LineGeometry()
state = HFE7200State(temperature_C=-110.0)
losses = LossCoefficients()
actual_id_m = id_from_od_wall(geometry.od_in, geometry.wall_in)

nominal = pressure_drop_hfe7200(4.0, geometry=geometry, state=state, losses=losses).iloc[0]
actual_envelope = pressure_drop_envelope(np.array([actual_id_m]), geometry=geometry, state=state).iloc[0]

assert np.isclose(actual_id_m * 1000.0, 7.747, atol=0.001)
assert np.isclose(float(hfe7200_density_kg_m3(-110.0)), 1734.386, atol=0.01)
assert np.isclose(float(hfe7200_kinematic_viscosity_cst(-110.0)), 28.3538516, atol=1.0e-6)
assert np.isclose(float(hfe7200_dynamic_viscosity_pa_s(-110.0)) * 1000.0, 49.1765232, atol=0.005)
assert np.isclose(nominal['Re'], 386.4321, rtol=5.0e-4)
assert np.isclose(nominal['dP_bar'], 1.4841255, rtol=5.0e-4)

print('Sanity checks passed.')
print(f'Actual tubing ID = {actual_id_m * 1000.0:.3f} mm')
print(f'HFE-7200 density at -110 °C = {nominal["density_kg_m3"]:.2f} kg/m^3')
print(f'HFE-7200 kinematic viscosity at -110 °C = {nominal["kinematic_viscosity_cSt"]:.2f} cSt')
print(f'HFE-7200 dynamic viscosity at -110 °C = {nominal["dynamic_viscosity_mPa_s"]:.2f} mPa s')
print(f'Re(4 L/min, actual ID) = {nominal["Re"]:.0f}')
print(f'Delta P(4 L/min, actual ID) = {nominal["dP_bar"]:.3f} bar = {nominal["dP_psi"]:.2f} psi')
print(f'Minor-K sensitivity at actual ID = {actual_envelope["dP_low_bar"]:.3f}..{actual_envelope["dP_high_bar"]:.3f} bar')

Sanity checks passed.
Actual tubing ID = 7.747 mm
HFE-7200 density at -110 °C = 1734.39 kg/m^3
HFE-7200 kinematic viscosity at -110 °C = 28.35 cSt
HFE-7200 dynamic viscosity at -110 °C = 49.18 mPa s
Re(4 L/min, actual ID) = 386
Delta P(4 L/min, actual ID) = 1.484 bar = 21.53 psi
Minor-K sensitivity at actual ID = 1.456..1.512 bar


In [4]:
component_rows = pd.DataFrame(
    {
        'quantity': [
            'Flow rate',
            'Temperature',
            'Inner diameter',
            'Density',
            'Kinematic viscosity',
            'Dynamic viscosity',
            'Mean velocity',
            'Reynolds number',
            'Darcy friction factor',
            'Pipe loss coefficient',
            'Minor loss coefficient',
            'Total loss coefficient',
            'Pressure drop',
            'Pressure drop',
            'Sensitivity band at actual ID',
        ],
        'value': [
            nominal['flow_L_min'],
            nominal['temperature_C'],
            nominal['diameter_m'] * 1000.0,
            nominal['density_kg_m3'],
            nominal['kinematic_viscosity_cSt'],
            nominal['dynamic_viscosity_mPa_s'],
            nominal['velocity_m_s'],
            nominal['Re'],
            nominal['darcy_f'],
            nominal['K_pipe'],
            nominal['K_minor'],
            nominal['K_total'],
            nominal['dP_bar'],
            nominal['dP_psi'],
            np.nan,
        ],
        'unit': [
            'L/min',
            'C',
            'mm',
            'kg/m^3',
            'cSt',
            'mPa s',
            'm/s',
            '-',
            '-',
            '-',
            '-',
            '-',
            'bar',
            'psi',
            f'{actual_envelope["dP_low_bar"]:.3f}..{actual_envelope["dP_high_bar"]:.3f} bar',
        ],
    }
)

display(component_rows)

,quantity,value,unit
0,Flow rate,4.000000,L/min
1,Temperature,-110.000000,C
2,Inner diameter,7.747000,mm
3,Density,1734.386000,kg/m^3
4,Kinematic viscosity,28.353852,cSt
5,Dynamic viscosity,49.176523,mPa s
6,Mean velocity,1.414333,m/s
7,Reynolds number,386.432142,-
8,Darcy friction factor,0.165618,-
9,Pipe loss coefficient,80.168627,-


In [5]:
flows = np.linspace(0.25, 4.5, 180)
flow_df = pressure_drop_hfe7200(flows, geometry=geometry, state=state, losses=losses)

fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
ax.plot(flow_df['flow_L_min'], flow_df['dP_bar'], color='#1f77b4', lw=2.2, label='Corrected model')
ax.scatter([nominal['flow_L_min']], [nominal['dP_bar']], color='#111827', zorder=4, label='4 L/min design point')
ax.axvline(4.0, color='#374151', ls='--', lw=1.0)
ax.set_xlabel('HFE-7200 flow rate [L/min]')
ax.set_ylabel('Pressure drop [bar]')
ax.set_title('Pressure drop through current HFE-7200 loop')
ax.legend(loc='upper left')

plt.show()

/tmp/ipykernel_1222386/3849773433.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
component_df = pressure_drop_hfe7200(flows, geometry=geometry, state=state, losses=losses)
rho = component_df['density_kg_m3'].to_numpy(float)
velocity = component_df['velocity_m_s'].to_numpy(float)
velocity_head_bar = rho * velocity**2 / 2.0 / PA_PER_BAR

component_df['dP_pipe_bar'] = component_df['K_pipe'] * velocity_head_bar
component_df['dP_minor_bar'] = component_df['K_minor'] * velocity_head_bar

fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
ax.plot(flows, component_df['dP_pipe_bar'], lw=2.0, label='Straight tubing')
ax.plot(flows, component_df['dP_minor_bar'], lw=2.0, label='Minor losses')
ax.plot(flows, component_df['dP_bar'], color='#111827', lw=2.3, label='Total')
ax.scatter([nominal['flow_L_min']], [nominal['dP_bar']], color='#111827', zorder=4)
ax.axvline(4.0, color='#374151', ls='--', lw=1.0)
ax.set_xlabel('HFE-7200 flow rate [L/min]')
ax.set_ylabel('Pressure drop [bar]')
ax.set_title('Pressure-drop contributions')
ax.legend(loc='upper left')

plt.show()

/tmp/ipykernel_1222386/398287209.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
ids_mm = np.linspace(4.0, 10.0, 220)
ids_m = ids_mm / 1000.0
nominal_diameter_df = pressure_drop_hfe7200(4.0, geometry=geometry, state=state, losses=losses, diameter_m=ids_m)

fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
ax.plot(ids_mm, nominal_diameter_df['dP_bar'], color='#1f77b4', lw=2.4, label='Nominal: -110 °C')
ax.axvline(actual_id_m * 1000.0, color='#111827', ls='--', lw=1.1, label='Actual ID = 7.747 mm')
ax.scatter([actual_id_m * 1000.0], [nominal['dP_bar']], color='#111827', zorder=4)
ax.set_xlabel('Tubing inner diameter [mm]')
ax.set_ylabel('Pressure drop at 4 L/min [bar]')
ax.set_xlim(ids_mm.min(), ids_mm.max())
ax.set_ylim(0.0, max(1.25, float(nominal_diameter_df['dP_bar'].max()) * 1.04))
ax.legend(loc='upper right')

plt.show()


/tmp/ipykernel_1222386/3193717494.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Main design number

With the corrected default assumptions used here:

- HFE-7200 at $-110^\circ\mathrm{C}$
- documented HFE-7200 viscosity, log-interpolated to $\nu = 28.35~\mathrm{cSt}$, so $\mu = 49.18~\mathrm{mPa\,s}$ using the 3M density law
- $Q = 4~\mathrm{L\,min^{-1}}$
- 3/8 in OD x 0.035 in wall tubing, $D_\mathrm{ID}=7.747~\mathrm{mm}$
- $L = 3.75~\mathrm{m}$
- six 90 deg bends, one contraction, one expansion, and one ball valve

The pressure drop is **1.484 bar** (**21.53 psi**) at the design point. The Reynolds number is **386**, so the flow is laminar, not transitional. As a separate model check, varying only the aggregate minor-loss coefficient by $\pm30\%$ gives **1.46..1.51 bar** at the actual ID.

This replaces the older notebook value based on $\mu=6~\mathrm{mPa\,s}$ and the extra laminar kinetic-energy correction. Standard pipe-friction and minor-loss pressure drops use the mean velocity head directly, so no additional $4/3$ multiplier is applied.

In [8]:
def actual_id_case(minor_loss_scale: float, label: str) -> dict[str, float | str]:
    row = pressure_drop_hfe7200(
        4.0,
        geometry=geometry,
        state=state,
        diameter_m=actual_id_m,
        minor_loss_scale=minor_loss_scale,
    ).iloc[0]
    return {
        'case': label,
        'temperature_C': row['temperature_C'],
        'minor_loss_scale': minor_loss_scale,
        'diameter_mm': row['diameter_m'] * 1000.0,
        'Re': row['Re'],
        'K_pipe': row['K_pipe'],
        'K_minor': row['K_minor'],
        'K_total': row['K_total'],
        'dP_bar': row['dP_bar'],
        'dP_psi': row['dP_psi'],
    }


summary = pd.DataFrame(
    [
        actual_id_case(1.0, 'Nominal corrected model'),
        actual_id_case(0.7, 'Sensitivity lower bound: minor K x 0.7'),
        actual_id_case(1.3, 'Sensitivity upper bound: minor K x 1.3'),
    ]
)

display(summary.round(4))

,case,temperature_C,minor_loss_scale,diameter_mm,Re,K_pipe,K_minor,K_total,dP_bar,dP_psi
0,Nominal corrected model,-110.0,1.0,7.747,386.4321,80.1686,5.3876,85.5562,1.4841,21.5254
1,Sensitivity lower bound: minor K x 0.7,-110.0,0.7,7.747,386.4321,80.1686,3.7713,83.9399,1.4561,21.1188
2,Sensitivity upper bound: minor K x 1.3,-110.0,1.3,7.747,386.4321,80.1686,7.0038,87.1724,1.5122,21.9321
